Preparación y validación del dataset para `train` y `test`. `*_gt.txt` es el GT oficial para tracking; `*_gt_whole.txt` queda separado para análisis. No se usan `RES_DET` ni `RES_MOT`.


In [1]:
from pathlib import Path
import csv,json,re,shutil
from collections import Counter
from PIL import Image
from IPython.display import display

ROOT=Path('/Users/angel/computer_vision/lab/practica4')
FRAMES=ROOT/'UAV-benchmark-M'
GT=ROOT/'UAV-benchmark-MOTD_v1.0'/'GT'
SPLITS={'train':ROOT/'M_attr'/'train','test':ROOT/'M_attr'/'test'}
CLASSES={1:'car',2:'truck',3:'bus'}
assert FRAMES.is_dir() and GT.is_dir() and all(p.is_dir() for p in SPLITS.values())
print('Splits:', {k:str(v) for k,v in SPLITS.items()})
print('GT oficial:',GT/'*_gt.txt')
print('GT análisis:',GT/'*_gt_whole.txt')


Splits: {'train': '/Users/angel/computer_vision/lab/practica4/M_attr/train', 'test': '/Users/angel/computer_vision/lab/practica4/M_attr/test'}
GT oficial: /Users/angel/computer_vision/lab/practica4/UAV-benchmark-MOTD_v1.0/GT/*_gt.txt
GT análisis: /Users/angel/computer_vision/lab/practica4/UAV-benchmark-MOTD_v1.0/GT/*_gt_whole.txt


## 1. Descubrimiento y validación

Se procesan por separado las 30 secuencias de `train` y las 20 de `test`. Se tolera `M0701_attr.txt` y `M0701 _attr.txt`. Se validan existencia, frames únicos, consecutivos y legibilidad. Frames sin vehículos son válidos y se reportan.


In [2]:
def sid(p):
    m=re.match(r'^\s*(M\d+)\s*_attr\.txt\s*$',p.name,re.I)
    assert m,p.name
    return m.group(1).upper()

def fnum(p):
    m=re.search(r'(\d+)',p.stem)
    return int(m.group(1)) if m else -1

def images(s):
    return sorted([p for p in (FRAMES/s).iterdir() if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'}],key=fnum) if (FRAMES/s).is_dir() else []

def discover(attr):
    seqs={}
    for p in sorted(attr.glob('*_attr.txt')):
        s=sid(p)
        assert s not in seqs,(attr,s)
        seqs[s]={'images':images(s),'gt':GT/f'{s}_gt.txt','whole':GT/f'{s}_gt_whole.txt','ignore':GT/f'{s}_gt_ignore.txt'}
    return seqs

seqs_by_split={split:discover(attr) for split,attr in SPLITS.items()}
assert len(seqs_by_split['train'])==30 and len(seqs_by_split['test'])==20, {k:len(v) for k,v in seqs_by_split.items()}
assert not (set(seqs_by_split['train']) & set(seqs_by_split['test']))
missing=[(split,s,k) for split,seqs in seqs_by_split.items() for s,d in seqs.items() for k in ('gt','whole','ignore') if not d[k].is_file() or (k=='gt' and not d[k].read_text().strip())]
assert not missing,missing

bad=[]
dims=Counter()
for split,seqs in seqs_by_split.items():
    for s,d in seqs.items():
        nums=[fnum(p) for p in d['images']]
        assert nums and not [x for x,n in Counter(nums).items() if n>1],(split,s,'frames duplicados')
        assert nums==list(range(min(nums),max(nums)+1)),(split,s,'frames no consecutivos')
        for p in d['images']:
            try:
                with Image.open(p) as im: im.verify()
                with Image.open(p) as im: dims[(im.size,im.mode)]+=1
            except Exception as e: bad.append((split,s,p.name,repr(e)))
assert not bad,bad[:3]
display([{'split':split,'sequence':s,'images':len(d['images']),'first':min(map(fnum,d['images'])),'last':max(map(fnum,d['images']))} for split,seqs in seqs_by_split.items() for s,d in sorted(seqs.items())])
print('Imágenes legibles:',sum(dims.values()),dict(dims))
print('Secuencias:', {k:len(v) for k,v in seqs_by_split.items()})


[{'split': 'train',
  'sequence': 'M0101',
  'images': 407,
  'first': 1,
  'last': 407},
 {'split': 'train',
  'sequence': 'M0201',
  'images': 1076,
  'first': 1,
  'last': 1076},
 {'split': 'train',
  'sequence': 'M0202',
  'images': 291,
  'first': 1,
  'last': 291},
 {'split': 'train',
  'sequence': 'M0204',
  'images': 350,
  'first': 1,
  'last': 350},
 {'split': 'train',
  'sequence': 'M0206',
  'images': 562,
  'first': 1,
  'last': 562},
 {'split': 'train',
  'sequence': 'M0207',
  'images': 885,
  'first': 1,
  'last': 885},
 {'split': 'train',
  'sequence': 'M0210',
  'images': 583,
  'first': 1,
  'last': 583},
 {'split': 'train',
  'sequence': 'M0301',
  'images': 325,
  'first': 1,
  'last': 325},
 {'split': 'train',
  'sequence': 'M0401',
  'images': 613,
  'first': 1,
  'last': 613},
 {'split': 'train',
  'sequence': 'M0402',
  'images': 410,
  'first': 1,
  'last': 410},
 {'split': 'train',
  'sequence': 'M0501',
  'images': 352,
  'first': 1,
  'last': 352},
 {'split

Imágenes legibles: 40735 {((1024, 540), 'RGB'): 38829, ((960, 540), 'RGB'): 1906}
Secuencias: {'train': 30, 'test': 20}


## 2. GT oficial, análisis, metadatos e ignore

La salida oficial conserva `frame,id,left,top,width,height,score,in-view,occlusion` de `*_gt.txt`. `in-view` y `occlusion` no se interpretan como visibilidad real.

La salida analítica filtra clases 1/2/3 de `*_gt_whole.txt` y escribe `conf=1,class=1/2/3,vis=-1` como marcadores; no afirma visibilidad. `out_of_view` y `occlusion` originales se guardan en JSON. Ignore queda separado: `class=-1` no basta para que `motmetrics` ignore cajas; hay que suministrarlas explícitamente.


In [3]:
def read(p):
    out=[]
    for n,line in enumerate(p.read_text(encoding='utf-8-sig').splitlines(),1):
        if line.strip():
            v=line.strip().split(',')
            assert len(v)==9,(p,n)
            out.append([int(float(x)) for x in v])
    return out

def write(p,rows):
    p.parent.mkdir(parents=True,exist_ok=True)
    with p.open('w',newline='') as h: csv.writer(h,lineterminator='\n').writerows(rows)

def validate(rows,frames,label):
    pairs=[(r[0],r[1]) for r in rows]
    dups=sorted(x for x,n in Counter(pairs).items() if n>1)
    noimg=sorted({r[0] for r in rows}-frames)
    positive=[]
    bounds=[]
    for r in rows:
        f,t,l,top,w,h=r[:6]
        if w<=0 or h<=0: positive.append(r)
        if l<0 or top<0 or l+w>1024 or top+h>540: bounds.append(r)
    return {'label':label,'rows':len(rows),'duplicate_frame_target_pairs':dups,'frames_without_image':noimg,'non_positive_boxes':len(positive),'out_of_bounds_boxes':len(bounds)}

OUTS={split:ROOT/'artifacts'/f'gt_mot_{split}' for split in SPLITS}
DIRS={split:{'official':out/'official_tracking_gt','analysis':out/'analysis_gt_whole','metadata':out/'metadata','ignore':out/'ignore'} for split,out in OUTS.items()}
for out in OUTS.values():
    if out.exists(): shutil.rmtree(out)
    for p in (out/'official_tracking_gt',out/'analysis_gt_whole',out/'metadata',out/'ignore'): p.mkdir(parents=True)

reports_by_split={}
for split,seqs in seqs_by_split.items():
    dirs=DIRS[split]
    reports=[]
    for s,d in sorted(seqs.items()):
        frames={fnum(p) for p in d['images']}
        official=sorted(read(d['gt']),key=lambda r:(r[0],r[1]))
        whole=sorted(read(d['whole']),key=lambda r:(r[0],r[1]))
        ignored=sorted(read(d['ignore']),key=lambda r:(r[0],r[1]))
        write(dirs['official']/f'{s}.txt',official)
        analysis=[[f,t,l,top,w,h,1,c,-1] for f,t,l,top,w,h,out,occ,c in whole if c in CLASSES]
        write(dirs['analysis']/f'{s}.txt',analysis)
        meta=[{'frame':f,'target_id':t,'left':l,'top':top,'width':w,'height':h,'out_of_view':out,'occlusion':occ,'class_id':c,'class_name':CLASSES.get(c,'other')} for f,t,l,top,w,h,out,occ,c in whole]
        (dirs['metadata']/f'{s}.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2))
        write(dirs['ignore']/f'{s}.txt',ignored)
        empty=sorted(frames-{r[0] for r in analysis})
        reports.append({'sequence':s,'official':validate(official,frames,'official_gt'),'analysis':validate(analysis,frames,'analysis_gt_whole_vehicles'),'ignore':validate(ignored,frames,'ignore'),'frames_without_vehicles':empty})
    reports_by_split[split]=reports

assert all(not r[k]['duplicate_frame_target_pairs'] for reports in reports_by_split.values() for r in reports for k in ('official','analysis','ignore'))
assert all(not r[k]['frames_without_image'] for reports in reports_by_split.values() for r in reports for k in ('official','analysis','ignore'))
assert all(r[k]['non_positive_boxes']==0 for reports in reports_by_split.values() for r in reports for k in ('official','analysis','ignore'))
for split,reports in reports_by_split.items():
    for r in reports:
        if r['frames_without_vehicles']: print(split,r['sequence'],'frames sin vehículos:',len(r['frames_without_vehicles']),r['frames_without_vehicles'][:20])
    print(split,'| Filas oficial:',sum(r['official']['rows'] for r in reports),'| análisis:',sum(r['analysis']['rows'] for r in reports),'| ignore:',sum(r['ignore']['rows'] for r in reports))
    print(split,'| Cajas fuera de límites reportadas:',sum(r[k]['out_of_bounds_boxes'] for r in reports for k in ('official','analysis','ignore')))
    display(reports)


train M0207 frames sin vehículos: 314 [572, 573, 574, 575, 576, 577, 578, 579, 580, 581, 582, 583, 584, 585, 586, 587, 588, 589, 590, 591]
train | Filas oficial: 422911 | análisis: 422911 | ignore: 49397
train | Cajas fuera de límites reportadas: 3


[{'sequence': 'M0101',
  'official': {'label': 'official_gt',
   'rows': 5414,
   'duplicate_frame_target_pairs': [],
   'frames_without_image': [],
   'non_positive_boxes': 0,
   'out_of_bounds_boxes': 0},
  'analysis': {'label': 'analysis_gt_whole_vehicles',
   'rows': 5414,
   'duplicate_frame_target_pairs': [],
   'frames_without_image': [],
   'non_positive_boxes': 0,
   'out_of_bounds_boxes': 0},
  'ignore': {'label': 'ignore',
   'rows': 670,
   'duplicate_frame_target_pairs': [],
   'frames_without_image': [],
   'non_positive_boxes': 0,
   'out_of_bounds_boxes': 0},
  'frames_without_vehicles': []},
 {'sequence': 'M0201',
  'official': {'label': 'official_gt',
   'rows': 6517,
   'duplicate_frame_target_pairs': [],
   'frames_without_image': [],
   'non_positive_boxes': 0,
   'out_of_bounds_boxes': 0},
  'analysis': {'label': 'analysis_gt_whole_vehicles',
   'rows': 6517,
   'duplicate_frame_target_pairs': [],
   'frames_without_image': [],
   'non_positive_boxes': 0,
   'out_

test M0203 frames sin vehículos: 12 [831, 832, 833, 834, 835, 836, 837, 838, 839, 840, 841, 842]
test | Filas oficial: 340906 | análisis: 375884 | ignore: 40990
test | Cajas fuera de límites reportadas: 61


[{'sequence': 'M0203',
  'official': {'label': 'official_gt',
   'rows': 5033,
   'duplicate_frame_target_pairs': [],
   'frames_without_image': [],
   'non_positive_boxes': 0,
   'out_of_bounds_boxes': 0},
  'analysis': {'label': 'analysis_gt_whole_vehicles',
   'rows': 5619,
   'duplicate_frame_target_pairs': [],
   'frames_without_image': [],
   'non_positive_boxes': 0,
   'out_of_bounds_boxes': 0},
  'ignore': {'label': 'ignore',
   'rows': 209,
   'duplicate_frame_target_pairs': [],
   'frames_without_image': [],
   'non_positive_boxes': 0,
   'out_of_bounds_boxes': 0},
  'frames_without_vehicles': [831,
   832,
   833,
   834,
   835,
   836,
   837,
   838,
   839,
   840,
   841,
   842]},
 {'sequence': 'M0205',
  'official': {'label': 'official_gt',
   'rows': 5219,
   'duplicate_frame_target_pairs': [],
   'frames_without_image': [],
   'non_positive_boxes': 0,
   'out_of_bounds_boxes': 0},
  'analysis': {'label': 'analysis_gt_whole_vehicles',
   'rows': 5458,
   'duplicate_f

## Verificación final

Usar únicamente `official_tracking_gt/` del split correspondiente para tracking. `analysis_gt_whole/` es análisis por clase; `metadata/` conserva `out_of_view` y `occlusion`; `ignore/` requiere integración explícita con el evaluador. `train` y `test` permanecen separados.


In [4]:
for split,dirs in DIRS.items():
    expected=30 if split=='train' else 20
    for key,p in dirs.items():
        files=list(p.glob('M*.*'))
        assert len(files)==expected,(split,key,p,len(files))
        print(split,p.relative_to(ROOT),len(files))
sample=DIRS['test']['official']/'M0203.txt'
print(sample)
print('\n'.join(sample.read_text().splitlines()[:3]))
assert all(len(x.split(','))==9 for x in sample.read_text().splitlines())
print('OK: ejecución completa; train y test están separados.')


train artifacts/gt_mot_train/official_tracking_gt 30
train artifacts/gt_mot_train/analysis_gt_whole 30
train artifacts/gt_mot_train/metadata 30
train artifacts/gt_mot_train/ignore 30
test artifacts/gt_mot_test/official_tracking_gt 20
test artifacts/gt_mot_test/analysis_gt_whole 20
test artifacts/gt_mot_test/metadata 20
test artifacts/gt_mot_test/ignore 20
/Users/angel/computer_vision/lab/practica4/artifacts/gt_mot_test/official_tracking_gt/M0203.txt
1,53,107,73,111,211,1,1,-1
1,54,187,90,56,74,1,1,-1
1,55,243,71,56,76,1,1,-1
OK: ejecución completa; train y test están separados.
